# Pneumonia Detection CNN — Fixed & Improved

This notebook fixes the bugs found in the original version and adds a few standard improvements:

1. **Fixed `subset` typo** (was `subsets`) — Keras silently ignores unknown kwargs, so `validation_split=0.2` was never applied. Both generators were loading the **full** training set (5216 images each), meaning "validation accuracy" was really just training accuracy.
2. **Fixed `shuffle` typo** (was `shuffles`) — this is the more serious bug. On the **test generator**, it meant `shuffle` defaulted to `True`. When a generator shuffles, `model.predict()` returns predictions in shuffled order, but `test_generator.classes` stays in original file order — so predictions and true labels were misaligned, which alone is enough to wreck a classification report.
3. Added light **data augmentation** to the training generator only (validation/test stay unaugmented).
4. Added **class weighting** to correct for the NORMAL/PNEUMONIA class imbalance.
5. Added **EarlyStopping**, **ModelCheckpoint**, and **ReduceLROnPlateau** callbacks, and increased epochs (with early stopping to prevent overfitting).
6. Switched to an explicit `Input()` layer to remove the `input_shape` deprecation warning.
7. Saved the trained model to disk at the end.

In [ ]:
!pip install numpy pandas tensorflow scikit-learn pillow

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

## 1. Build the training dataframe

In [ ]:
train_dir = r'D:\archive\chest_xray\train/'
categories = ['PNEUMONIA', 'NORMAL']
filepaths = []
labels = []

for category in categories:
    folder = os.path.join(train_dir, category)
    for fname in os.listdir(folder):
        filepaths.append(f"{category}/{fname}")
        labels.append(category)

df = pd.DataFrame({'Filename': filepaths, 'Label': labels})
df['Label'].value_counts()

## 2. Data generators

Two separate `ImageDataGenerator` instances are used so that augmentation only applies to
training data, while both share the same `validation_split` and dataframe so the train/validation
split is non-overlapping and consistent.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

# No augmentation on the validation slice -- only rescaling, so validation
# metrics reflect performance on unaltered images.
valid_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    directory=train_dir,
    x_col='Filename',
    y_col='Label',
    subset='training',       # FIXED: was 'subsets' (typo, silently ignored by Keras)
    batch_size=32,
    seed=42,
    shuffle=True,             # FIXED: was 'shuffles'
    class_mode='binary',
    target_size=(150, 150)
)

valid_generator = valid_datagen.flow_from_dataframe(
    dataframe=df,
    directory=train_dir,
    x_col='Filename',
    y_col='Label',
    subset='validation',     # FIXED: was 'subsets'
    batch_size=32,
    seed=42,
    shuffle=True,             # FIXED: was 'shuffles'
    class_mode='binary',
    target_size=(150, 150)
)

## 3. Class weights

The dataset is imbalanced (more PNEUMONIA images than NORMAL), which biased the original model toward predicting PNEUMONIA. Class weights correct for this during training.

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))
class_weight_dict

## 4. Model

In [ ]:
model = Sequential([
    Input(shape=(150, 150, 3)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(3, 3),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(3, 3),
    Flatten(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Callbacks

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint('best_pneumonia_model.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

## 6. Train

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=15,
    validation_data=valid_generator,
    validation_steps=valid_generator.samples // valid_generator.batch_size,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

## 7. Build the test dataframe

In [ ]:
test_dir = r'D:\archive\chest_xray\test/'
test_filepaths = []
test_labels = []

for category in categories:
    folder = os.path.join(test_dir, category)
    for fname in os.listdir(folder):
        test_filepaths.append(f"{category}/{fname}")
        test_labels.append(category)

test_df = pd.DataFrame({'Filename': test_filepaths, 'Label': test_labels})

## 8. Test generator

`shuffle=False` is essential here so that the order of predictions matches the order of `test_generator.classes`.

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=test_dir,
    x_col='Filename',
    y_col='Label',
    batch_size=32,
    shuffle=False,            # FIXED: was 'shuffles' (typo). Must be False so
                               # predictions align with test_generator.classes.
    class_mode='binary',
    target_size=(150, 150)
)

## 9. Evaluate on the test set

In [ ]:
pred_probs = model.predict(test_generator)
preds = (pred_probs > 0.5).astype(int).flatten()
true_labels = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

print(classification_report(true_labels, preds, target_names=class_labels))
print(confusion_matrix(true_labels, preds))

## 10. Save the trained model

In [ ]:
model.save('pneumonia_cnn_model.keras')